# Exhaustive search for trees with exactly three Q-main eigenvalues (SageMath)

This notebook performs the exhaustive computational search over all trees of order $n \leq 35$ for trees with exactly three Q-main eigenvalues.


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import math
import subprocess
import os
import json
import time
import sys
import io
import warnings
from collections import defaultdict
from operator import mul as _mul

# SageMath 兼容性：确保使用 Python 原生类型
import builtins
py_int = builtins.int
py_float = builtins.float

# 多进程数量（留 2 核给系统和 notebook 本身）
NUM_WORKERS = max(1, os.cpu_count() - 2)

%matplotlib inline

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'PingFang SC', 'Heiti TC', 'STHeiti']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print(f"可用 CPU 核心: {os.cpu_count()}, 将使用 {NUM_WORKERS} 个工作进程")

In [ ]:
# ===================================================================
# 核心功能: 整数精确判定 Q-主特征值个数
# ===================================================================
#
# 数学原理：
#   Q-主特征值个数 = dim(Krylov 子空间 K(Q, j))
#   其中 Q = A + D (无符号 Laplacian), j = 全1向量
#
#   构造 W = [j, Qj, Q²j, ..., Q^k j]  (n × (k+1) 矩阵)
#   Q-主特征值个数恰为 k  ⟺  rank(W) = k
#
# 整数 Gram 矩阵 + Hankel 结构：
#   - W 的所有列都是整数向量
#   - Gram 矩阵 G = W^T W 有 Hankel 结构：G[i,j] 仅依赖 i+j
#   - 对于 k=3, G 是 4×4 Hankel 矩阵，由 h0..h6 共 7 个参数决定
#   - 其中 h0 = n, h1 = 4(n-1) 对所有 n 阶树都是常数
#   - 仅需计算 5 个内积 (h2..h6)，而非原始的 10 个
#
# 性能优化（V5 Hankel）：
#   1. 无邻接表：直接用 parent array 计算 Q*v（省去 n 个 list 创建）
#   2. v1 = 2*deg 快捷计算，用 v1>>1 恢复 deg
#   3. Hankel 恒等式：g01=4(n-1), g02=g11, g03=g12, g13=g22（省 4 个内积）
#   4. sum(map(mul, vi, vj)) 替代 generator（C级内积速度）
#   5. 行列式全内联
# ===================================================================


def check_q_main_k3_from_parent(parent_array):
    """
    从 parent array 判定树是否有恰好 3 个 Q-主特征值。

    参数:
        parent_array: list[int], 长度 n, parent[0]=0,
                      1-indexed: 实际父节点 = parent[i] - 1

    返回:
        bool: 恰有 3 个 Q-主特征值返回 True
    """
    n = len(parent_array)
    if n < 3:
        return False

    # ---- v1 = 2*deg，无需构造 adj 和 deg 数组 ----
    v1 = [0] * n
    for i in range(1, n):
        p = parent_array[i] - 1
        v1[i] += 2
        v1[p] += 2

    # ---- v2 = Q*v1，直接用 parent array（deg = v1>>1）----
    v2 = [(v1[i] >> 1) * v1[i] for i in range(n)]
    for i in range(1, n):
        p = parent_array[i] - 1
        v2[i] += v1[p]
        v2[p] += v1[i]

    # ---- v3 = Q*v2 ----
    v3 = [(v1[i] >> 1) * v2[i] for i in range(n)]
    for i in range(1, n):
        p = parent_array[i] - 1
        v3[i] += v2[p]
        v3[p] += v2[i]

    # ---- Hankel Gram 矩阵 ----
    # h0 = n, h1 = 4(n-1) 是常数，无需计算
    # h2 = <v1,v1> = g11 = g02
    # h3 = <v1,v2> = g12 = g03
    # h4 = <v2,v2> = g22 = g13
    # h5 = <v2,v3> = g23
    # h6 = <v3,v3> = g33
    h0 = n
    h1 = (n - 1) << 2  # 4*(n-1)
    h2 = sum(map(_mul, v1, v1))
    h3 = sum(map(_mul, v1, v2))
    h4 = sum(map(_mul, v2, v2))
    h5 = sum(map(_mul, v2, v3))
    h6 = sum(map(_mul, v3, v3))

    # ---- 4×4 Hankel 行列式 ----
    d4 = (h0*(h2*(h4*h6 - h5*h5) - h3*(h3*h6 - h5*h4) + h4*(h3*h5 - h4*h4))
        - h1*(h1*(h4*h6 - h5*h5) - h3*(h2*h6 - h5*h3) + h4*(h2*h5 - h4*h3))
        + h2*(h1*(h3*h6 - h5*h4) - h2*(h2*h6 - h5*h3) + h4*(h2*h4 - h3*h3))
        - h3*(h1*(h3*h5 - h4*h4) - h2*(h2*h5 - h4*h3) + h3*(h2*h4 - h3*h3)))

    if d4 != 0:
        return False   # rank = 4

    # ---- 3×3 主子式检查（rank = 3?）----
    # skip 0: [[h2,h3,h4],[h3,h4,h5],[h4,h5,h6]]
    if (h2*(h4*h6 - h5*h5) - h3*(h3*h6 - h5*h4) + h4*(h3*h5 - h4*h4)) != 0:
        return True
    # skip 1: [[h0,h2,h3],[h2,h4,h5],[h3,h5,h6]]
    if (h0*(h4*h6 - h5*h5) - h2*(h2*h6 - h5*h3) + h3*(h2*h5 - h4*h3)) != 0:
        return True
    # skip 2: [[h0,h1,h3],[h1,h2,h4],[h3,h4,h6]]
    if (h0*(h2*h6 - h4*h4) - h1*(h1*h6 - h4*h3) + h3*(h1*h4 - h2*h3)) != 0:
        return True
    # skip 3: [[h0,h1,h2],[h1,h2,h3],[h2,h3,h4]]
    if (h0*(h2*h4 - h3*h3) - h1*(h1*h4 - h3*h2) + h2*(h1*h3 - h2*h2)) != 0:
        return True

    return False   # rank ≤ 2


def check_q_main_eigenvalue_count(G, k):
    """
    兼容旧接口：从 networkx 图对象判定 Q-主特征值个数是否恰为 k。
    仅用于绘图阶段的二次验证或兼容旧代码调用。
    """
    n = G.order()
    if n < k:
        return False

    if k == 3:
        nodes = sorted(G.nodes())
        node_map = {v: i for i, v in enumerate(nodes)}
        parent = [0] * n
        from collections import deque
        visited = [False] * n
        visited[0] = True
        queue = deque([0])
        while queue:
            u = queue.popleft()
            for v_orig in G.neighbors(nodes[u]):
                v = node_map[v_orig]
                if not visited[v]:
                    visited[v] = True
                    parent[v] = u + 1
                    queue.append(v)
        return check_q_main_k3_from_parent(parent)

    # 对于其他 k 值，保留通用的 numpy 方法
    nodes = list(G.nodes())
    A = nx.to_numpy_array(G, nodelist=nodes, dtype=np.int64)
    deg_vec = np.array([G.degree(v) for v in nodes], dtype=np.int64)
    Q = A.copy()
    for i in range(n):
        Q[i, i] += deg_vec[i]

    j = np.ones(n, dtype=np.int64)
    vecs = [j.copy()]
    cur = j.copy()
    for _ in range(k):
        cur = Q @ cur
        vecs.append(cur.copy())

    dim = k + 1
    Gram = [[int(np.dot(vecs[i], vecs[j])) for j in range(dim)] for i in range(dim)]

    import fractions
    mat = [[fractions.Fraction(Gram[i][j]) for j in range(dim)] for i in range(dim)]
    rank = 0
    for col in range(dim):
        pivot = None
        for row in range(rank, dim):
            if mat[row][col] != 0:
                pivot = row
                break
        if pivot is None:
            continue
        mat[rank], mat[pivot] = mat[pivot], mat[rank]
        for row in range(dim):
            if row != rank and mat[row][col] != 0:
                factor = mat[row][col] / mat[rank][col]
                for c in range(dim):
                    mat[row][c] -= factor * mat[rank][c]
        rank += 1

    return rank == k


print("核心判定函数已加载：V5 Hankel 优化（无邻接表 + Hankel 恒等式 + 5 内积）")

In [ ]:
# ===================================================================
# 树生成: 使用 gentreeg -p 输出 parent array
# ===================================================================
#
# 优化说明：
#   旧方法：gentreeg -s → sparse6 字符串 → nx.read_sparse6() → networkx Graph
#   新方法：gentreeg -p → parent array 文本 → split() → list[int]
#
#   parent array 格式：每行 n 个空格分隔的整数（1-indexed）
#     parent[0] = 0 表示节点 0 是根
#     parent[i] (i>=1) 是节点 i 的父节点编号（1-indexed）
#     实际边集 = {(i, parent[i]-1) | i = 1, ..., n-1}
#
# 正确性论证：
#   1. gentreeg 枚举所有非同构无标号树，-p 和 -s 只是输出格式不同，
#      树的集合完全相同（已通过计数验证）
#   2. parent array 直接给出树的结构，无需中间转换
#   3. 每棵树恰好输出一行，解析无歧义
#
# 性能提升：
#   - 省去 sparse6 解码（位运算 + 字节解析）
#   - 省去 networkx Graph 对象创建（Python 字典、边集等）
#   - 省去 nx.to_numpy_array() 的二次遍历
#   - 字符串 split + int 转换是 Python 中最快的文本解析方式之一
# ===================================================================

def generate_parent_arrays(n, res=None, mod=None):
    """
    流式生成阶数为 n 的所有树的 parent array。

    参数:
        n: 树的阶数
        res, mod: 可选，gentreeg 的分片参数（用于多进程并行）
                  当指定时，只生成第 res 片（共 mod 片）

    yields:
        list[int]: 每棵树的 parent array
    """
    n_py = py_int(n)
    cmd = ["gentreeg", "-p", "-q", str(n_py)]
    if res is not None and mod is not None:
        cmd.append(f"{py_int(res)}/{py_int(mod)}")

    try:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            bufsize=py_int(65536)
        )
    except FileNotFoundError:
        raise RuntimeError(
            "未找到 gentreeg，请先确认 nauty/gentreeg 已安装并加入 PATH"
        )

    bad_lines = 0
    line_count = 0

    try:
        for line in proc.stdout:
            line = line.strip()
            if not line:
                continue

            line_count += 1

            try:
                parent = list(map(py_int, line.split()))
            except ValueError:
                bad_lines += 1
                continue

            if len(parent) != n_py:
                bad_lines += 1
                continue

            yield parent

        stderr_output = proc.stderr.read()
        return_code = py_int(proc.wait())

        if return_code != 0:
            raise RuntimeError(
                f"gentreeg 执行失败，返回码 {return_code}。\n"
                f"stderr: {stderr_output.strip()}"
            )

        if bad_lines > 0:
            raise RuntimeError(
                f"n={n_py} 枚举完成，但有 {bad_lines}/{line_count} 行解析失败！"
            )

    finally:
        for stream in [proc.stdout, proc.stderr]:
            try:
                if stream:
                    stream.close()
            except:
                pass
        try:
            if proc.poll() is None:
                proc.kill()
                proc.wait()
        except:
            pass


def parent_array_to_nx_graph(parent_array):
    """将 parent array 转换为 networkx 图（仅在需要绘图时使用）"""
    n = len(parent_array)
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for i in range(1, n):
        G.add_edge(i, parent_array[i] - 1)  # 1-indexed → 0-indexed
    return G


def diameter_from_parent(parent_array):
    """从 parent array 直接计算树的直径（两次 BFS）"""
    n = len(parent_array)
    if n <= 1:
        return 0

    # 构造邻接表
    adj = [[] for _ in range(n)]
    for i in range(1, n):
        p = parent_array[i] - 1   # 1-indexed → 0-indexed
        adj[i].append(p)
        adj[p].append(i)

    # BFS 找最远节点
    def bfs_farthest(start):
        from collections import deque
        dist = [-1] * n
        dist[start] = 0
        queue = deque([start])
        farthest = start
        max_dist = 0
        while queue:
            u = queue.popleft()
            for v in adj[u]:
                if dist[v] == -1:
                    dist[v] = dist[u] + 1
                    if dist[v] > max_dist:
                        max_dist = dist[v]
                        farthest = v
                    queue.append(v)
        return farthest, max_dist

    far1, _ = bfs_farthest(0)
    _, diam = bfs_farthest(far1)
    return diam


print("树生成函数已加载：使用 parent array 格式（无 networkx 开销）")

In [ ]:
# ===================================================================
# 多进程并行分析 + C worker V4 + 中断续跑 + 结果自动保存
# ===================================================================
#
# V4 优化（相对 V3 C worker）：
#   1. double 预筛选：先用 double 计算 d4，若明显非零直接跳过 __int128，
#      对 100% 的非 k=3 树生效（sound：绝不漏判）
#   2. Block I/O：512KB 缓冲区 + 自定义行扫描替代 fgets
#   3. 直接管道：fork/exec 替代 popen（省去 shell 开销）
#   4. __builtin_expect 分支预测提示（d4≠0 路径）
#   5. 继承 V3 全部优化：-O3 -march=native、CSE、int 数组、快速解析
#
# V1→V4 总加速历程（n=25 单worker 受控对比）：
#   V1 (-O2, long long):     35.5s
#   V3 (V1 + 编译器/CSE等):  18.0s → V1 的 50.7%
#   V4 (V3 + 预筛选/IO/管道): 15.3s → V1 的 43.1%
#   多进程下（8 worker, n=25）: V3 11.3s → V4c 6.6s → V1 的 57%
#
# 正确性保证：
#   1. double 预筛选是 SOUND 的：仅当 |d4_double| > 误差上界时才跳过，
#      对真正 d4=0 的树一定进入精确路径（无假阴性）
#   2. 精确路径与 V3 完全一致（__int128 + CSE）
#   3. 已在 n=6..25 全量验证与 V3 结果完全一致
#   4. 任何 worker 失败都会导致整阶报错
# ===================================================================

import tempfile
import hashlib


# ---- SageMath 兼容的 JSON 编码器 ----
class _SageJSONEncoder(json.JSONEncoder):
    def default(self, obj):
        try:
            return py_int(obj)
        except (TypeError, ValueError):
            pass
        try:
            return py_float(obj)
        except (TypeError, ValueError):
            pass
        return super().default(obj)


# ---- 结果保存目录 ----
RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath("Q-all-fix-next.ipynb")), "results_q_main_eigenvalues")


def _ensure_results_dir(results_dir):
    """确保结果目录可用；兼容悬空符号链接。"""
    if os.path.islink(results_dir) and not os.path.exists(results_dir):
        link_target = os.path.realpath(results_dir)
        os.makedirs(link_target, exist_ok=True)
        return results_dir
    if os.path.exists(results_dir) and not os.path.isdir(results_dir):
        raise NotADirectoryError(f"结果路径已存在但不是目录: {results_dir}")
    os.makedirs(results_dir, exist_ok=True)
    return results_dir


# ---- C worker V4 源码 ----
_C_WORKER_SOURCE = r'''
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <math.h>
#define MAXN 64
#define LIKELY(x)   __builtin_expect(!!(x), 1)
#define UNLIKELY(x) __builtin_expect(!!(x), 0)

/* ---- Block I/O: 512KB buffer + in-place newline scanning ---- */
#define BUFSZ (512*1024)
static char iobuf[BUFSZ + 4096];
static int io_pos, io_len, io_fd;
static inline void io_init(int fd) { io_fd = fd; io_pos = 0; io_len = 0; }
static inline char* io_readline(void) {
    while (1) {
        for (int i = io_pos; i < io_len; i++) {
            if (iobuf[i] == '\n') {
                iobuf[i] = '\0';
                char *line = iobuf + io_pos;
                io_pos = i + 1;
                return line;
            }
        }
        int lo = io_len - io_pos;
        if (lo > 0 && io_pos > 0) memmove(iobuf, iobuf + io_pos, lo);
        io_pos = 0; io_len = lo;
        int nr = read(io_fd, iobuf + io_len, BUFSZ);
        if (nr <= 0) {
            if (io_len > 0) { iobuf[io_len] = '\0'; char *l = iobuf; io_len = 0; return l; }
            return NULL;
        }
        io_len += nr;
    }
}

/* ---- Fast inline integer parse (0..99) ---- */
static inline int parse_line(const char *p, int *pa, int n) {
    for (int i = 0; i < n; i++) {
        while (*p == ' ') p++;
        if (UNLIKELY(*p < '0' || *p > '9')) return i;
        int v = *p++ - '0';
        if (*p >= '0' && *p <= '9') v = v * 10 + (*p++ - '0');
        pa[i] = v;
    }
    return n;
}

/* ---- Q-main k=3 check: double pre-screening + exact fallback ----
 * Phase 1 (double): compute d4 in double. If |d4| >> error bound,
 *   d4 is definitely nonzero → return 0. Handles 100% of non-k3 trees.
 * Phase 2 (exact): __int128 CSE determinant for the <0.0001% ambiguous cases.
 */
static int check_k3(const int *pa, int n) {
    if (UNLIKELY(n < 3)) return 0;
    int v1[MAXN], v2[MAXN], v3[MAXN];
    int i, p;
    memset(v1, 0, n * sizeof(int));
    for (i = 1; i < n; i++) { p = pa[i]-1; v1[i] += 2; v1[p] += 2; }
    for (i = 0; i < n; i++) v2[i] = (v1[i] >> 1) * v1[i];
    for (i = 1; i < n; i++) { p = pa[i]-1; v2[i] += v1[p]; v2[p] += v1[i]; }
    for (i = 0; i < n; i++) v3[i] = (v1[i] >> 1) * v2[i];
    for (i = 1; i < n; i++) { p = pa[i]-1; v3[i] += v2[p]; v3[p] += v2[i]; }

    /* Phase 1: double pre-screening */
    {
        double dh2=0, dh3=0, dh4=0, dh5=0, dh6=0;
        for (i = 0; i < n; i++) {
            double dv1=v1[i], dv2=v2[i], dv3=v3[i];
            dh2+=dv1*dv1; dh3+=dv1*dv2; dh4+=dv2*dv2;
            dh5+=dv2*dv3; dh6+=dv3*dv3;
        }
        double dh0=n, dh1=4.0*(n-1);
        double dp46=dh4*dh6-dh5*dh5, dp36=dh3*dh6-dh4*dh5, dp35=dh3*dh5-dh4*dh4;
        double dp26=dh2*dh6-dh3*dh5, dp25=dh2*dh5-dh3*dh4, dp24=dh2*dh4-dh3*dh3;
        double ds0=dh2*dp46-dh3*dp36+dh4*dp35;
        double dc1=dh1*dp46-dh3*dp26+dh4*dp25;
        double dc2=dh1*dp36-dh2*dp26+dh4*dp24;
        double dc3=dh1*dp35-dh2*dp25+dh3*dp24;
        double dd4=dh0*ds0-dh1*dc1+dh2*dc2-dh3*dc3;
        double abs_sum=fabs(dh0*ds0)+fabs(dh1*dc1)+fabs(dh2*dc2)+fabs(dh3*dc3);
        if (LIKELY(fabs(dd4) > abs_sum * 256.0 * 2.3e-16)) return 0;
    }

    /* Phase 2: exact __int128 + CSE */
    long long h0=n, h1=4LL*(n-1);
    long long h2=0, h3=0, h4=0, h5=0, h6=0;
    for (i = 0; i < n; i++) {
        h2+=(long long)v1[i]*v1[i]; h3+=(long long)v1[i]*v2[i];
        h4+=(long long)v2[i]*v2[i]; h5+=(long long)v2[i]*v3[i];
        h6+=(long long)v3[i]*v3[i];
    }
    typedef __int128 I;
    I ip46=(I)h4*h6-(I)h5*h5, ip36=(I)h3*h6-(I)h4*h5, ip35=(I)h3*h5-(I)h4*h4;
    I ip26=(I)h2*h6-(I)h3*h5, ip25=(I)h2*h5-(I)h3*h4, ip24=(I)h2*h4-(I)h3*h3;
    I s0=(I)h2*ip46-(I)h3*ip36+(I)h4*ip35;
    I c01=(I)h1*ip46-(I)h3*ip26+(I)h4*ip25;
    I c02=(I)h1*ip36-(I)h2*ip26+(I)h4*ip24;
    I c03=(I)h1*ip35-(I)h2*ip25+(I)h3*ip24;
    I d4=(I)h0*s0-(I)h1*c01+(I)h2*c02-(I)h3*c03;
    if (LIKELY(d4 != 0)) return 0;
    if (s0 != 0) return 1;
    if ((I)h0*ip46-(I)h2*ip26+(I)h3*ip25 != 0) return 1;
    I ip2644=(I)h2*h6-(I)h4*h4;
    I ip16=(I)h1*h6-(I)h3*h4;
    I ip14=(I)h1*h4-(I)h2*h3;
    if ((I)h0*ip2644-(I)h1*ip16+(I)h3*ip14 != 0) return 1;
    I ip13=(I)h1*h3-(I)h2*h2;
    if ((I)h0*ip24-(I)h1*ip14+(I)h2*ip13 != 0) return 1;
    return 0;
}

static int diameter(const int *pa, int n) {
    if (n <= 1) return 0;
    int adj_head[MAXN], adj_next[2*MAXN], adj_to[2*MAXN], ec = 0;
    memset(adj_head, -1, n * sizeof(int));
    for (int i = 1; i < n; i++) {
        int p = pa[i]-1;
        adj_to[ec]=p; adj_next[ec]=adj_head[i]; adj_head[i]=ec++;
        adj_to[ec]=i; adj_next[ec]=adj_head[p]; adj_head[p]=ec++;
    }
    int queue[MAXN], dist[MAXN], far, mx, head, tail;
    memset(dist,-1,n*sizeof(int)); dist[0]=0; queue[0]=0; head=0; tail=1; far=0; mx=0;
    while(head<tail){int u=queue[head++];for(int e=adj_head[u];e!=-1;e=adj_next[e]){int v=adj_to[e];if(dist[v]==-1){dist[v]=dist[u]+1;if(dist[v]>mx){mx=dist[v];far=v;}queue[tail++]=v;}}}
    memset(dist,-1,n*sizeof(int)); dist[far]=0; queue[0]=far; head=0; tail=1; mx=0;
    while(head<tail){int u=queue[head++];for(int e=adj_head[u];e!=-1;e=adj_next[e]){int v=adj_to[e];if(dist[v]==-1){dist[v]=dist[u]+1;if(dist[v]>mx)mx=dist[v];queue[tail++]=v;}}}
    return mx;
}

int main(int argc, char **argv) {
    if (argc != 4) { fprintf(stderr,"Usage: %s n res mod\n",argv[0]); return 1; }
    int n=atoi(argv[1]), res=atoi(argv[2]), mod=atoi(argv[3]);
    if (n>MAXN) { fprintf(stderr,"n=%d > MAXN=%d\n",n,MAXN); return 1; }

    /* Direct pipe to gentreeg (no shell overhead) */
    int pipefd[2];
    if (pipe(pipefd) == -1) { perror("pipe"); return 1; }
    pid_t pid = fork();
    if (pid == -1) { perror("fork"); return 1; }
    if (pid == 0) {
        close(pipefd[0]);
        dup2(pipefd[1], STDOUT_FILENO);
        close(pipefd[1]);
        char sn[16], sres[32];
        snprintf(sn, sizeof(sn), "%d", n);
        snprintf(sres, sizeof(sres), "%d/%d", res, mod);
        execlp("gentreeg", "gentreeg", "-p", "-q", sn, sres, (char*)NULL);
        _exit(127);
    }
    close(pipefd[1]);

    io_init(pipefd[0]);
    int pa[MAXN];
    long long count = 0; int k3_count = 0;
    #define MAX_K3 1000
    int k3_pa[MAX_K3][MAXN], k3_diam[MAX_K3];
    char *line;
    while ((line = io_readline()) != NULL) {
        if (UNLIKELY(line[0] == '\0')) continue;
        if (UNLIKELY(parse_line(line, pa, n) != n)) continue;
        count++;
        if (UNLIKELY(check_k3(pa, n))) {
            if (k3_count<MAX_K3){memcpy(k3_pa[k3_count],pa,n*sizeof(int));k3_diam[k3_count]=diameter(pa,n);}
            k3_count++;
        }
    }
    close(pipefd[0]);
    int status;
    waitpid(pid, &status, 0);
    if (!WIFEXITED(status) || WEXITSTATUS(status) != 0) {
        fprintf(stderr,"gentreeg failed\n"); return 1;
    }
    printf("{\"count\":%lld,\"k3_trees\":[",count);
    for (int i=0;i<k3_count&&i<MAX_K3;i++){
        if(i>0)printf(",");printf("{\"parent\":[");
        for(int j=0;j<n;j++){if(j>0)printf(",");printf("%d",k3_pa[i][j]);}
        printf("],\"diameter\":%d}",k3_diam[i]);
    }
    printf("]}\n");
    return 0;
}
'''

# ---- Python V5 fallback worker ----
_WORKER_SCRIPT = r'''
import subprocess, sys, json
from collections import deque
from operator import mul

def check_k3(pa):
    n = len(pa)
    if n < 3: return False
    v1 = [0]*n
    for i in range(1, n):
        p = pa[i]-1; v1[i]+=2; v1[p]+=2
    v2 = [(v1[i]>>1)*v1[i] for i in range(n)]
    for i in range(1, n):
        p = pa[i]-1; v2[i]+=v1[p]; v2[p]+=v1[i]
    v3 = [(v1[i]>>1)*v2[i] for i in range(n)]
    for i in range(1, n):
        p = pa[i]-1; v3[i]+=v2[p]; v3[p]+=v2[i]
    h0=n; h1=(n-1)<<2
    h2=sum(map(mul,v1,v1)); h3=sum(map(mul,v1,v2))
    h4=sum(map(mul,v2,v2)); h5=sum(map(mul,v2,v3)); h6=sum(map(mul,v3,v3))
    d4=(h0*(h2*(h4*h6-h5*h5)-h3*(h3*h6-h5*h4)+h4*(h3*h5-h4*h4))
       -h1*(h1*(h4*h6-h5*h5)-h3*(h2*h6-h5*h3)+h4*(h2*h5-h4*h3))
       +h2*(h1*(h3*h6-h5*h4)-h2*(h2*h6-h5*h3)+h4*(h2*h4-h3*h3))
       -h3*(h1*(h3*h5-h4*h4)-h2*(h2*h5-h4*h3)+h3*(h2*h4-h3*h3)))
    if d4 != 0: return False
    if (h2*(h4*h6-h5*h5)-h3*(h3*h6-h5*h4)+h4*(h3*h5-h4*h4)) != 0: return True
    if (h0*(h4*h6-h5*h5)-h2*(h2*h6-h5*h3)+h3*(h2*h5-h4*h3)) != 0: return True
    if (h0*(h2*h6-h4*h4)-h1*(h1*h6-h4*h3)+h3*(h1*h4-h2*h3)) != 0: return True
    if (h0*(h2*h4-h3*h3)-h1*(h1*h4-h3*h2)+h2*(h1*h3-h2*h2)) != 0: return True
    return False

def diam(pa):
    n = len(pa)
    if n <= 1: return 0
    adj = [[] for _ in range(n)]
    for i in range(1, n):
        p = pa[i]-1; adj[i].append(p); adj[p].append(i)
    def bfs(s):
        dist = [-1]*n; dist[s] = 0; q = deque([s]); far = s; mx = 0
        while q:
            u = q.popleft()
            for v in adj[u]:
                if dist[v] == -1:
                    dist[v] = dist[u]+1
                    if dist[v] > mx: mx = dist[v]; far = v
                    q.append(v)
        return far, mx
    f, _ = bfs(0); _, d = bfs(f)
    return d

n, res, mod = int(sys.argv[1]), int(sys.argv[2]), int(sys.argv[3])
cmd = ["gentreeg", "-p", "-q", str(n), f"{res}/{mod}"]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=65536)
count = 0; k3_trees = []; bad_lines = 0
for line in proc.stdout:
    line = line.strip()
    if not line: continue
    try:
        parent = list(map(int, line.split()))
    except ValueError:
        bad_lines += 1; continue
    if len(parent) != n:
        bad_lines += 1; continue
    count += 1
    if check_k3(parent):
        k3_trees.append({"parent": parent, "diameter": diam(parent)})
stderr_out = proc.stderr.read()
rc = proc.wait()
if rc != 0:
    print(f"ERROR: gentreeg failed rc={rc}. stderr: {stderr_out.strip()}", file=sys.stderr)
    sys.exit(1)
if bad_lines > 0:
    print(f"ERROR: {bad_lines} bad lines out of {count+bad_lines}", file=sys.stderr)
    sys.exit(1)
json.dump({"count": count, "k3_trees": k3_trees}, sys.stdout)
'''


def _compile_c_worker():
    """编译 C worker V4，返回可执行文件路径。
    使用源码 SHA256 哈希缓存：如果已有二进制且源码未变，跳过编译。
    编译失败返回 None。
    """
    c_file = os.path.join(tempfile.gettempdir(), '_q_worker_v4.c')
    bin_file = os.path.join(tempfile.gettempdir(), '_q_worker_v4')
    hash_file = os.path.join(tempfile.gettempdir(), '_q_worker_v4.sha256')

    src_hash = hashlib.sha256(_C_WORKER_SOURCE.encode()).hexdigest()

    # 检查缓存：二进制存在 + 哈希匹配 → 跳过编译
    if os.path.isfile(bin_file) and os.path.isfile(hash_file):
        try:
            with open(hash_file, 'r') as f:
                cached_hash = f.read().strip()
            if cached_hash == src_hash:
                check = subprocess.run(
                    [bin_file, "4", "0", "1"],
                    capture_output=True, text=True, timeout=10
                )
                if check.returncode == 0:
                    d = json.loads(check.stdout)
                    if d['count'] == 2:
                        print("C worker V4 已缓存，跳过编译（源码未变）")
                        return bin_file
        except Exception:
            pass

    with open(c_file, 'w') as f:
        f.write(_C_WORKER_SOURCE)

    try:
        result = subprocess.run(
            ['cc', '-O3', '-march=native', '-flto', '-funroll-loops',
             '-o', bin_file, c_file, '-lm'],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode == 0:
            check = subprocess.run(
                [bin_file, "4", "0", "1"],
                capture_output=True, text=True, timeout=10
            )
            if check.returncode == 0:
                d = json.loads(check.stdout)
                if d['count'] == 2:
                    with open(hash_file, 'w') as f:
                        f.write(src_hash)
                    print("C worker V4 编译成功（-O3 -march=native + double预筛选 + BlockIO + 直接管道）")
                    return bin_file
            print("C worker V4 健康检查失败")
        else:
            result = subprocess.run(
                ['cc', '-O3', '-flto', '-o', bin_file, c_file, '-lm'],
                capture_output=True, text=True, timeout=30
            )
            if result.returncode == 0:
                check = subprocess.run(
                    [bin_file, "4", "0", "1"],
                    capture_output=True, text=True, timeout=10
                )
                if check.returncode == 0:
                    d = json.loads(check.stdout)
                    if d['count'] == 2:
                        with open(hash_file, 'w') as f:
                            f.write(src_hash)
                        print("C worker V4 编译成功（-O3 -flto，无 -march=native）")
                        return bin_file
            print(f"C worker V4 编译失败: {result.stderr[:200]}")
        return None
    except FileNotFoundError:
        print("未找到 C 编译器，使用 Python worker")
        return None
    except Exception as e:
        print(f"C worker V4 编译异常: {e}")
        return None


# ---- 全局编译一次 ----
_C_WORKER_BIN = _compile_c_worker()


def _run_workers_parallel(n, num_workers):
    """启动 num_workers 个并行 worker 子进程。

    优先使用 C worker V4，失败时回退到 Python V5 worker。
    如果任何 worker 失败，立即抛出异常。
    """
    use_c = _C_WORKER_BIN is not None

    if not use_c:
        worker_file = os.path.join(tempfile.gettempdir(), '_q_main_worker.py')
        with open(worker_file, 'w') as f:
            f.write(_WORKER_SCRIPT)

    procs = []
    for res in range(py_int(num_workers)):
        if use_c:
            cmd = [_C_WORKER_BIN, str(py_int(n)), str(py_int(res)), str(py_int(num_workers))]
        else:
            cmd = [sys.executable, worker_file, str(py_int(n)), str(py_int(res)), str(py_int(num_workers))]
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        procs.append(p)

    total = 0
    k3_trees = []
    errors = []
    for i, p in enumerate(procs):
        stdout, stderr = p.communicate()
        if p.returncode != 0:
            errors.append(f"Worker {i} 失败 (返回码 {p.returncode}): {stderr[:300]}")
            continue
        try:
            result = json.loads(stdout)
            total += result['count']
            for t in result['k3_trees']:
                k3_trees.append((t['parent'], t['diameter']))
        except Exception as e:
            errors.append(f"Worker {i} 输出解析失败: {e}")

    if errors:
        raise RuntimeError(
            f"n={n} 的并行计算有 {len(errors)} 个 worker 失败，"
            f"结果不完整，不允许保存或使用:\n" + "\n".join(errors)
        )

    return total, k3_trees


def _save_results_for_n(n, k_value, total_trees, k3_trees, elapsed, results_dir):
    """将阶数 n 的结果保存到 JSON"""
    results_dir = _ensure_results_dir(results_dir)

    result = {
        'n': py_int(n),
        'k_value': py_int(k_value),
        'total_trees': py_int(total_trees),
        'k3_count': py_int(len(k3_trees)),
        'elapsed_seconds': py_float(round(elapsed, 2)),
        'trees_per_second': py_float(round(total_trees / elapsed, 1)) if elapsed > 0 else 0,
        'k3_trees': [
            {'parent_array': [py_int(x) for x in pa], 'diameter': py_int(d)}
            for pa, d in k3_trees
        ],
        'diameter_distribution': {},
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
    }

    for _, d in k3_trees:
        key = str(py_int(d))
        result['diameter_distribution'][key] = \
            py_int(result['diameter_distribution'].get(key, 0) + 1)

    filepath = os.path.join(results_dir, f"n{py_int(n)}_k{py_int(k_value)}.json")
    with open(filepath, 'w') as f:
        json.dump(result, f, indent=py_int(2), ensure_ascii=False, cls=_SageJSONEncoder)

    return filepath


def _load_completed_orders(k_value, results_dir):
    """加载已完成的阶数集合"""
    completed = {}
    if not os.path.exists(results_dir):
        return completed
    for fname in os.listdir(results_dir):
        if fname.startswith('n') and fname.endswith(f'_k{py_int(k_value)}.json'):
            try:
                with open(os.path.join(results_dir, fname)) as f:
                    data = json.load(f)
                    completed[data['n']] = data
            except:
                pass
    return completed


def analyze_single_order(n, k_value, num_workers):
    """分析单个阶数 n 的所有树。仅支持 k=3。"""
    if py_int(k_value) != 3:
        raise NotImplementedError(f"当前仅支持 k=3，不支持 k={k_value}。")

    start_time = time.time()

    if py_int(num_workers) <= 1 or py_int(n) <= 14:
        total = 0
        k3_trees = []
        for parent in generate_parent_arrays(n):
            total += 1
            if check_q_main_k3_from_parent(parent):
                d = diameter_from_parent(parent)
                k3_trees.append((parent, d))
        elapsed = time.time() - start_time
        return total, k3_trees, elapsed

    total, k3_trees = _run_workers_parallel(n, num_workers)
    elapsed = time.time() - start_time
    return total, k3_trees, elapsed


def analyze_trees_optimized(k_value=3, min_order=6, max_order=30,
                            max_save_per_diameter=100,
                            num_workers=None, results_dir=None):
    """优化版主分析函数。"""
    if num_workers is None:
        num_workers = NUM_WORKERS
    if results_dir is None:
        results_dir = RESULTS_DIR
    results_dir = _ensure_results_dir(results_dir)

    worker_type = "C worker V4" if _C_WORKER_BIN else "Python V5 worker"

    print(f"{'='*70}")
    print(f"分析 Q-主特征值个数 k={k_value} 的树")
    print(f"搜索阶数范围: n={min_order} 到 n={max_order}")
    print(f"判定方法: 整数精确 Hankel Gram 矩阵秩")
    print(f"树格式: parent array（无 networkx 开销）")
    print(f"并行模式: {num_workers} 个 {worker_type}")
    print(f"结果保存: {results_dir}")
    print(f"{'='*70}\n")

    completed = _load_completed_orders(k_value, results_dir)
    if completed:
        print(f"检测到已完成的阶数: {sorted(completed.keys())}")
        print(f"将跳过这些阶数。\n")

    global_stats = {
        'total_trees': 0,
        'k_trees_by_diameter': defaultdict(py_int),
        'saved_trees_by_diameter': defaultdict(list),
    }

    for n_done, data in sorted(completed.items()):
        if min_order <= n_done <= max_order:
            global_stats['total_trees'] += data['total_trees']
            for tree_info in data['k3_trees']:
                d = tree_info['diameter']
                global_stats['k_trees_by_diameter'][d] += 1
                if len(global_stats['saved_trees_by_diameter'][d]) < max_save_per_diameter:
                    global_stats['saved_trees_by_diameter'][d].append(
                        (tree_info['parent_array'], d, n_done)
                    )

    for n in range(py_int(min_order), py_int(max_order) + 1):
        print(f"\n{'─'*70}")

        if n in completed:
            data = completed[n]
            print(f"阶数 n={n}: 已有保存结果，跳过")
            print(f"    - 该阶树总数: {data['total_trees']:,}")
            print(f"    - 满足 k={k_value}: {data['k3_count']}")
            continue

        print(f"正在处理阶数 n={n} 的所有树...")

        try:
            total, k3_trees, elapsed = analyze_single_order(n, k_value, num_workers)

            global_stats['total_trees'] += total
            for pa, d in k3_trees:
                global_stats['k_trees_by_diameter'][d] += 1
                if len(global_stats['saved_trees_by_diameter'][d]) < max_save_per_diameter:
                    global_stats['saved_trees_by_diameter'][d].append((pa, d, n))

            filepath = _save_results_for_n(n, k_value, total, k3_trees, elapsed, results_dir)

            print(f"  完成 n={n}")
            print(f"    - 该阶树总数: {py_int(total):,}")
            print(f"    - 满足 k={k_value}: {py_int(len(k3_trees))}")
            print(f"    - 用时: {py_float(elapsed):.2f} 秒")
            if total > 0:
                speed = py_float(total) / py_float(elapsed) if elapsed > 0 else 0.0
                ratio = py_float(len(k3_trees)) / py_float(total) * 100.0
                print(f"    - 速度: {speed:,.0f} 树/秒")
                print(f"    - 满足比例: {ratio:.6f}%")
            if k3_trees:
                diam_dist = defaultdict(py_int)
                for _, d in k3_trees:
                    diam_dist[py_int(d)] += 1
                print(f"    - 直径分布: {dict(sorted(diam_dist.items()))}")
            print(f"    - 已保存: {filepath}")

        except Exception as e:
            print(f"  错误: {e}")
            import traceback
            traceback.print_exc()
            break

    print(f"\n{'='*70}")
    print("分析完成！")
    print(f"{'='*70}")
    print(f"\nQ-主特征值个数 k={k_value} 的树的直径分布：")
    print(f"{'直径':>6} | {'数量':>10}")
    print("-" * 30)
    for d in sorted(global_stats['k_trees_by_diameter'].keys()):
        count = global_stats['k_trees_by_diameter'][d]
        print(f"{py_int(d):6d} | {py_int(count):10d}")
    total_k = sum(global_stats['k_trees_by_diameter'].values())
    print(f"\n总计: {py_int(total_k)} 棵 k={k_value} 树")
    print(f"总共检查: {py_int(global_stats['total_trees']):,} 棵树\n")

    return global_stats


print("分析框架已加载：C worker V4 + Python V5 fallback + 中断续跑 + 编译缓存")

In [6]:
def hierarchy_pos(G, root=None, width=1., vert_gap=0.2, vert_loc=0, xcenter=0.5):
    """为树创建层次布局"""
    if not nx.is_tree(G):
        raise TypeError('Graph must be a tree')

    if root is None and len(G) > 0:
        eccentricity = nx.eccentricity(G)
        center_nodes = [v for v in G.nodes() if eccentricity[v] == min(eccentricity.values())]
        root = center_nodes[0]
    elif root is None:
        root = list(G.nodes())[0]

    def _hierarchy_pos(G, root, width=1., vert_gap=0.2, vert_loc=0, xcenter=0.5,
                       pos=None, parent=None, parsed=None):
        if pos is None:
            pos = {root: (xcenter, vert_loc)}
        else:
            pos[root] = (xcenter, vert_loc)

        if parsed is None:
            parsed = [root]
        else:
            parsed.append(root)

        neighbors = list(G.neighbors(root))
        if parent is not None and parent in neighbors:
            neighbors.remove(parent)

        if len(neighbors) != 0:
            dx = width / len(neighbors)
            nextx = xcenter - width / 2 - dx / 2
            for neighbor in neighbors:
                nextx += dx
                pos = _hierarchy_pos(
                    G, neighbor, width=dx, vert_gap=vert_gap,
                    vert_loc=vert_loc - vert_gap, xcenter=nextx,
                    pos=pos, parent=root, parsed=parsed
                )
        return pos

    return _hierarchy_pos(G, root, width, vert_gap, vert_loc, xcenter)


def plot_single_tree(G, ax, show_labels=False):
    """绘制单棵树"""
    n = G.order()

    if n == 0:
        ax.set_title("空图", fontsize=8)
        ax.axis('off')
        return

    tree_id = G.graph.get('id', 'N/A')
    diameter = G.graph.get('diameter', '?')
    k_value = G.graph.get('k_value', '?')

    title_line1 = f"{tree_id} (n={n}, d={diameter}, Q-main k={k_value})"
    title = title_line1

    if diameter == 5:
        try:
            center_nodes = nx.center(G)
            if len(center_nodes) == 2:
                u, v = center_nodes[0], center_nodes[1]

                c1, r1, a = 0, 0, []
                neighbors_u = list(G.neighbors(u))
                neighbors_u.remove(v)

                for n_u in neighbors_u:
                    if G.degree(n_u) == 1:
                        c1 += 1
                    else:
                        r1 += 1
                        a.append(G.degree(n_u) - 1)

                c2, r2, b = 0, 0, []
                neighbors_v = list(G.neighbors(v))
                neighbors_v.remove(u)

                for n_v in neighbors_v:
                    if G.degree(n_v) == 1:
                        c2 += 1
                    else:
                        r2 += 1
                        b.append(G.degree(n_v) - 1)

                a.sort()
                b.sort()
                title_line2 = f"r1={r1}, c1={c1}, a={a}"
                title_line3 = f"r2={r2}, c2={c2}, b={b}"
                title = f"{title_line1}\n{title_line2}\n{title_line3}"
        except:
            pass

    pos = None
    try:
        pos = hierarchy_pos(G, width=5.0, vert_gap=0.5)
    except:
        try:
            pos = nx.kamada_kawai_layout(G)
        except:
            pos = nx.spring_layout(
                G,
                k=2.0 / np.sqrt(n) if n > 1 else 1,
                iterations=200,
                seed=42
            )

    node_size = max(50, min(300, 500 // n))
    edge_width = max(0.8, min(2.5, 30 / n))
    font_size = max(6, min(10, 100 // n))

    try:
        nx.draw_networkx_edges(
            G, pos, ax=ax, edge_color='#666666',
            width=edge_width, alpha=0.6,
            connectionstyle='arc3,rad=0.1'
        )
    except (TypeError, ValueError):
        nx.draw_networkx_edges(
            G, pos, ax=ax, edge_color='#666666',
            width=edge_width, alpha=0.6
        )

    nx.draw_networkx_nodes(
        G, pos, ax=ax, node_color='#87CEEB',
        node_size=node_size, edgecolors='#4682B4',
        linewidths=2, alpha=0.9
    )

    if show_labels and n <= 20:
        nx.draw_networkx_labels(
            G, pos, ax=ax, font_size=font_size,
            font_weight='bold', font_color='#000000'
        )

    ax.set_title(title, fontsize=10, pad=10, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

    if pos:
        x_values = [coord[0] for coord in pos.values()]
        y_values = [coord[1] for coord in pos.values()]
        x_margin = (max(x_values) - min(x_values)) * 0.1 or 1
        y_margin = (max(y_values) - min(y_values)) * 0.1 or 1
        ax.set_xlim(min(x_values) - x_margin, max(x_values) + x_margin)
        ax.set_ylim(min(y_values) - y_margin, max(y_values) + y_margin)


def plot_trees_by_diameter(trees_dict, k_value, trees_per_page=48):
    """按直径分别绘制树"""
    figures = []

    for diameter in sorted(trees_dict.keys()):
        tree_list = trees_dict[diameter]
        if not tree_list:
            continue

        if diameter >= 6:
            tree_list = tree_list[:50]

        total_trees = len(tree_list)
        num_pages = math.ceil(total_trees / trees_per_page)

        print(f"\n绘制直径={diameter}的树（共{total_trees}棵，分{num_pages}页）...")

        for page in range(num_pages):
            start_idx = page * trees_per_page
            end_idx = min((page + 1) * trees_per_page, total_trees)
            page_trees = tree_list[start_idx:end_idx]
            num_in_page = len(page_trees)

            cols = min(8, int(np.ceil(np.sqrt(num_in_page * 1.2))))
            rows = math.ceil(num_in_page / cols)

            fig = plt.figure(figsize=(cols * 4, rows * 4))
            gs = fig.add_gridspec(rows, cols, hspace=0.4, wspace=0.3)

            if diameter >= 6 and total_trees > 50:
                title_suffix = "（仅展示前50棵）"
            else:
                title_suffix = ""

            if num_pages == 1:
                title = f"Q-主特征值个数 k={k_value}、直径={diameter} 的所有树 (共{total_trees}棵){title_suffix}"
            else:
                title = f"Q-主特征值个数 k={k_value}、直径={diameter} 的树 (第{page+1}/{num_pages}页){title_suffix}"

            fig.suptitle(title, fontsize=16, fontweight='bold', y=0.995)

            for i, G in enumerate(page_trees):
                row = i // cols
                col = i % cols
                ax = fig.add_subplot(gs[row, col])
                plot_single_tree(G, ax, show_labels=False)

            if num_pages == 1:
                filename = f"Qk{k_value}_diameter{diameter}_trees_all.png"
            else:
                filename = f"Qk{k_value}_diameter{diameter}_trees_page{page+1}_of_{num_pages}.png"

            fig.savefig(filename, dpi=300, bbox_inches='tight', facecolor='white')
            print(f"  ✓ 已保存: {filename}")

            figures.append(fig)

    return figures

In [ ]:
# =======================================================
# 主程序 - 优化版
# =======================================================

print("\n" + "=" * 70)
print(" " * 10 + "树的 Q-主特征值分析程序（优化版）")
print("=" * 70)

# ============ 配置参数 ============
K_VALUE = 3                  # Q-主特征值个数
MIN_ORDER = 6               # 最小阶数
MAX_ORDER = 35              # 最大阶数
MAX_SAVE_PER_DIAMETER = 100  # 每个直径最多保存的树数量
TREES_PER_PAGE = 48          # 每页显示的树数量
# ==================================

print(f"\n配置:")
print(f"  - Q-主特征值个数 k: {K_VALUE}")
print(f"  - 搜索阶数范围: {MIN_ORDER} - {MAX_ORDER}")
print(f"  - 判定方法: 整数精确 Gram 矩阵秩")
print(f"  - 树格式: parent array")
print(f"  - 并行进程数: {NUM_WORKERS}")
print(f"  - 每个直径最多保存: {MAX_SAVE_PER_DIAMETER} 棵")

try:
    stats = analyze_trees_optimized(
        k_value=K_VALUE,
        min_order=MIN_ORDER,
        max_order=MAX_ORDER,
        max_save_per_diameter=MAX_SAVE_PER_DIAMETER,
    )

    # 绘图：将 parent array 转换为 networkx 图
    if stats['saved_trees_by_diameter']:
        print("\n开始绘制树...")

        trees_dict = defaultdict(list)
        for d in sorted(stats['saved_trees_by_diameter'].keys()):
            for pa, diam, n_order in stats['saved_trees_by_diameter'][d]:
                G = parent_array_to_nx_graph(pa)
                G.graph['id'] = f'n{n_order}_d{diam}_Qk{K_VALUE}_{len(trees_dict[d])+1}'
                G.graph['diameter'] = diam
                G.graph['k_value'] = K_VALUE
                trees_dict[d].append(G)

        figures = plot_trees_by_diameter(trees_dict, K_VALUE, TREES_PER_PAGE)

        print(f"\n{'='*70}")
        print(f"绘制完成！共生成 {len(figures)} 个图形")
        print(f"{'='*70}\n")
    else:
        print(f"\n没有找到 Q-主特征值个数为 {K_VALUE} 的树")

except KeyboardInterrupt:
    print("\n\n用户中断程序（已完成的阶数结果已自动保存，重启可续跑）")
except Exception as e:
    print(f"\n\n程序出错: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 70)
print("程序执行完成！")
print("=" * 70 + "\n")